# 개선우선순위 추천 - 단일 포인트 SHAP 확인

이 노트북은 한 포인트를 선택해 두 모델의 SHAP 기여도를 확인하고, 개선우선순위를 계산한다.

- 모델 1: 사고발생확률 `p`
- 모델 2: 조건부 위험도 `r`
- 최종점수: `p * r`
- 추천 기준: 두 모델에서 공통으로 양수 기여한 개선 후보 feature

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
import warnings
from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', '{:.6f}'.format)

PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'src').exists() and (candidate / 'data').exists()
)
SRC_DIR = PROJECT_ROOT / 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

KOREAN_FONT_CANDIDATES = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
    '/System/Library/Fonts/AppleSDGothicNeo.ttc',
    '/Library/Fonts/AppleGothic.ttf',
]
for font_path in KOREAN_FONT_CANDIDATES:
    if Path(font_path).exists():
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
        break
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

from silverwalk_ai.features.preprocessing import OriginalTrainPreprocessor, TARGET_COLUMN
from scripts.explain.recommend_improvements import RECOMMENDATION_MAPPING, normalize_positive

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'matplotlib font: {plt.rcParams["font.family"]}')

I0000 00:00:1780798085.889578  109000 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780798085.917644  109000 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780798086.657265  109000 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


ModuleNotFoundError: No module named 'scripts'

## 1. 설정

`POINT_ID = None`이면 최종 위험도 점수가 가장 높은 포인트 1개를 자동 선택한다.
특정 포인트를 보고 싶으면 `POINT_ID`에 값을 입력한다.

In [ ]:
RAW_PATH = PROJECT_ROOT / 'data' / 'original_train_data' / 'seoul_road_points.csv'
PREDICTIONS_PATH = PROJECT_ROOT / 'artifacts' / 'predictions' / 'two_stage_zero_risk_predictions.csv'
PREPROCESSOR_PATH = PROJECT_ROOT / 'artifacts' / 'preprocessors' / 'original_train_preprocessor.joblib'
CLASSIFIER_MODEL_PATH = PROJECT_ROOT / 'artifacts' / 'models' / 'mlp_accident_classifier.keras'
REGRESSOR_MODEL_PATH = PROJECT_ROOT / 'artifacts' / 'models' / 'mlp_positive_risk_regressor.keras'
PREDICT_SCRIPT_PATH = PROJECT_ROOT / 'scripts' / 'predict' / 'predict_two_stage_zero_risk.py'

POINT_ID = None  # 예: 12345. None이면 최종위험도점수_percent가 가장 높은 포인트 선택
PERCENT_THRESHOLD = 10.0
BACKGROUND_SIZE = 128
SHAP_NSAMPLES = 100
TOP_FEATURES_TO_PLOT = 15
TOP_RECOMMENDATIONS = 5
RANDOM_STATE = 42

path_status = pd.DataFrame([
    {'name': 'raw data', 'path': RAW_PATH.relative_to(PROJECT_ROOT), 'exists': RAW_PATH.exists()},
    {'name': 'predictions', 'path': PREDICTIONS_PATH.relative_to(PROJECT_ROOT), 'exists': PREDICTIONS_PATH.exists()},
    {'name': 'preprocessor', 'path': PREPROCESSOR_PATH.relative_to(PROJECT_ROOT), 'exists': PREPROCESSOR_PATH.exists()},
    {'name': 'classifier model', 'path': CLASSIFIER_MODEL_PATH.relative_to(PROJECT_ROOT), 'exists': CLASSIFIER_MODEL_PATH.exists()},
    {'name': 'regressor model', 'path': REGRESSOR_MODEL_PATH.relative_to(PROJECT_ROOT), 'exists': REGRESSOR_MODEL_PATH.exists()},
])
display(path_status)

## 2. 최종 예측 CSV 준비

`two_stage_zero_risk_predictions.csv`가 없으면 예측 스크립트를 실행해 생성한다.

In [ ]:
if not PREDICTIONS_PATH.exists():
    required = [RAW_PATH, PREPROCESSOR_PATH, CLASSIFIER_MODEL_PATH, REGRESSOR_MODEL_PATH, PREDICT_SCRIPT_PATH]
    missing = [path for path in required if not path.exists()]
    if missing:
        missing_text = '\n'.join(str(path.relative_to(PROJECT_ROOT)) for path in missing)
        raise FileNotFoundError(missing_text)

    command = [
        sys.executable,
        str(PREDICT_SCRIPT_PATH),
        '--input', str(RAW_PATH),
        '--classifier-model-path', str(CLASSIFIER_MODEL_PATH),
        '--regressor-model-path', str(REGRESSOR_MODEL_PATH),
        '--preprocessor-path', str(PREPROCESSOR_PATH),
        '--output', str(PREDICTIONS_PATH),
        '--device', 'auto',
    ]
    print('최종 예측 CSV가 없어 생성합니다:')
    print(' '.join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print(f'기존 예측 CSV 사용: {PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}')


## 3. 포인트 선택

기본 조건은 지도 시각화 대상과 동일하게 `위험도_actual = 0`이고 `최종위험도점수_percent > 10`인 포인트다.

In [ ]:
predictions = pd.read_csv(PREDICTIONS_PATH)
required_columns = [
    'POINT_ID',
    '위도',
    '경도',
    '위험도_actual',
    '사고발생확률_p',
    '조건부위험도_r',
    '최종위험도점수',
    '최종위험도점수_percent',
]
missing = [column for column in required_columns if column not in predictions.columns]
if missing:
    raise ValueError(f'예측 CSV에 필요한 컬럼이 없습니다: {missing}')

candidate_points = predictions[
    (predictions['위험도_actual'] == 0)
    & (predictions['최종위험도점수_percent'] > PERCENT_THRESHOLD)
].copy()
candidate_points = candidate_points.sort_values('최종위험도점수_percent', ascending=False)
if candidate_points.empty:
    raise ValueError('조건에 맞는 포인트가 없습니다. PERCENT_THRESHOLD를 낮춰보십시오.')

if POINT_ID is None:
    selected_prediction = candidate_points.iloc[0].copy()
else:
    matched = predictions[predictions['POINT_ID'] == POINT_ID]
    if matched.empty:
        raise ValueError(f'POINT_ID를 찾을 수 없습니다: {POINT_ID}')
    selected_prediction = matched.iloc[0].copy()

selected_point_id = selected_prediction['POINT_ID']
print(f'선택된 POINT_ID: {selected_point_id}')
display(selected_prediction[required_columns].to_frame('value'))

display(candidate_points.head(10)[required_columns])

## 4. 선택 포인트와 SHAP 배경 데이터 준비

SHAP 기준 배경 데이터는 `위험도 = 0` 포인트에서 샘플링한다.

In [ ]:
raw = pd.read_csv(RAW_PATH)
preprocessor = OriginalTrainPreprocessor.load(PREPROCESSOR_PATH)

selected_raw = raw[raw['POINT_ID'] == selected_point_id].copy()
if selected_raw.empty:
    raise ValueError(f'원본 데이터에서 POINT_ID를 찾을 수 없습니다: {selected_point_id}')

zero_risk_raw = raw[raw[TARGET_COLUMN] == 0].copy()
background_raw = zero_risk_raw.sample(
    n=min(BACKGROUND_SIZE, len(zero_risk_raw)),
    random_state=RANDOM_STATE,
)

selected_features_frame = preprocessor.transform_features(selected_raw)
background_features_frame = preprocessor.transform_features(background_raw)
feature_columns = preprocessor.feature_columns

selected_features = selected_features_frame[feature_columns].to_numpy(dtype=np.float32)
background_features = background_features_frame[feature_columns].to_numpy(dtype=np.float32)

print(f'feature count: {len(feature_columns)}')
print(f'background samples: {len(background_features)}')
display(selected_raw[['POINT_ID', '위도', '경도', TARGET_COLUMN]].head())

## 5. 모델 로드와 SHAP 계산

두 모델 각각에 대해 선택 포인트의 SHAP 값을 계산한다.

In [ ]:
import shap

classifier_model = tf.keras.models.load_model(CLASSIFIER_MODEL_PATH)
regressor_model = tf.keras.models.load_model(REGRESSOR_MODEL_PATH)

classifier_pred = float(classifier_model.predict(selected_features, verbose=0).reshape(-1)[0])
regressor_pred_log1p = float(regressor_model.predict(selected_features, verbose=0).reshape(-1)[0])
regressor_pred = float(np.clip(np.expm1(regressor_pred_log1p), 0, None))
final_score = classifier_pred * regressor_pred

print(f'모델 1 사고발생확률 p: {classifier_pred:.6f}')
print(f'모델 2 조건부위험도 log1p r: {regressor_pred_log1p:.6f}')
print(f'모델 2 조건부위험도 r: {regressor_pred:.6f}')
print(f'최종점수 p*r: {final_score:.6f}')


def shap_values_for_one_point(model: tf.keras.Model, background: np.ndarray, target: np.ndarray, nsamples: int) -> np.ndarray:
    explainer = shap.GradientExplainer(model, background)
    values = explainer.shap_values(target, nsamples=nsamples)
    if isinstance(values, list):
        values = values[0]
    values = np.asarray(values)
    if values.ndim == 3 and values.shape[-1] == 1:
        values = values[:, :, 0]
    if values.ndim != 2 or values.shape[0] != 1:
        raise ValueError(f'Unexpected SHAP shape: {values.shape}')
    return values[0]

classifier_shap = shap_values_for_one_point(
    classifier_model,
    background_features,
    selected_features,
    nsamples=SHAP_NSAMPLES,
)
regressor_shap = shap_values_for_one_point(
    regressor_model,
    background_features,
    selected_features,
    nsamples=SHAP_NSAMPLES,
)

print('SHAP 계산 완료')
print('model1 shap shape:', classifier_shap.shape)
print('model2 shap shape:', regressor_shap.shape)

## 6. 한 포인트의 SHAP 분포 그래프

양수 SHAP는 해당 feature가 위험 예측을 높이는 방향으로 기여했다는 뜻이고, 음수 SHAP는 낮추는 방향으로 기여했다는 뜻이다.

In [ ]:
shap_frame = pd.DataFrame({
    'feature': feature_columns,
    'model1_shap': classifier_shap,
    'model2_shap': regressor_shap,
    'model1_abs_shap': np.abs(classifier_shap),
    'model2_abs_shap': np.abs(regressor_shap),
})
shap_frame['combined_abs_shap'] = shap_frame['model1_abs_shap'] + shap_frame['model2_abs_shap']
shap_frame['model1_positive'] = shap_frame['model1_shap'] > 0
shap_frame['model2_positive'] = shap_frame['model2_shap'] > 0
shap_frame['common_positive'] = shap_frame['model1_positive'] & shap_frame['model2_positive']

top_shap = shap_frame.sort_values('combined_abs_shap', ascending=False).head(TOP_FEATURES_TO_PLOT).copy()
display(top_shap[['feature', 'model1_shap', 'model2_shap', 'common_positive']])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

model1_plot = top_shap.sort_values('model1_shap')
axes[0].barh(
    model1_plot['feature'],
    model1_plot['model1_shap'],
    color=np.where(model1_plot['model1_shap'] >= 0, '#dc2626', '#2563eb'),
    alpha=0.85,
)
axes[0].axvline(0, color='black', linewidth=1)
axes[0].set_title('모델 1 SHAP 분포: 사고발생확률 p')
axes[0].set_xlabel('SHAP value')

model2_plot = top_shap.sort_values('model2_shap')
axes[1].barh(
    model2_plot['feature'],
    model2_plot['model2_shap'],
    color=np.where(model2_plot['model2_shap'] >= 0, '#dc2626', '#2563eb'),
    alpha=0.85,
)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_title('모델 2 SHAP 분포: 조건부위험도 r')
axes[1].set_xlabel('SHAP value')

plt.tight_layout()
plt.show()

## 7. 개선우선순위 계산

추천 후보 feature 중 두 모델에서 모두 양수 SHAP인 항목만 개선 후보로 본다.

In [ ]:
recommendation_features = [feature for feature in RECOMMENDATION_MAPPING if feature in feature_columns]
recommendation_indices = np.array([feature_columns.index(feature) for feature in recommendation_features])

classifier_reco = classifier_shap[recommendation_indices]
regressor_reco = regressor_shap[recommendation_indices]
classifier_norm = normalize_positive(classifier_reco.reshape(1, -1))[0]
regressor_norm = normalize_positive(regressor_reco.reshape(1, -1))[0]
common_positive = (classifier_reco > 0) & (regressor_reco > 0)
recommendation_scores = (classifier_norm + regressor_norm) * common_positive

recommendation_frame = pd.DataFrame({
    '개선추천': [RECOMMENDATION_MAPPING[feature] for feature in recommendation_features],
    'feature': recommendation_features,
    'model1_shap': classifier_reco,
    'model2_shap': regressor_reco,
    'model1_positive': classifier_reco > 0,
    'model2_positive': regressor_reco > 0,
    'common_positive': common_positive,
    'recommendation_score': recommendation_scores,
})
recommendation_frame = recommendation_frame.sort_values('recommendation_score', ascending=False).reset_index(drop=True)
recommendation_frame['rank'] = np.where(
    recommendation_frame['recommendation_score'] > 0,
    np.arange(1, len(recommendation_frame) + 1),
    np.nan,
)

display(recommendation_frame)

top_recommendations = recommendation_frame[recommendation_frame['recommendation_score'] > 0].head(TOP_RECOMMENDATIONS)
if top_recommendations.empty:
    print('두 모델에서 공통으로 양수 기여한 개선 추천 항목이 없습니다.')
else:
    print('개선우선순위')
    for idx, row in top_recommendations.iterrows():
        print(
            f"{int(row['rank'])}. {row['개선추천']} "
            f"(feature={row['feature']}, score={row['recommendation_score']:.6f}, "
            f"model1_shap={row['model1_shap']:.6f}, model2_shap={row['model2_shap']:.6f})"
        )

## 8. 개선우선순위 그래프

추천 후보 feature만 대상으로 점수를 시각화한다.

In [ ]:
plot_reco = recommendation_frame.sort_values('recommendation_score')
fig, ax = plt.subplots(figsize=(12, 6))
colors = np.where(plot_reco['recommendation_score'] > 0, '#dc2626', '#94a3b8')
ax.barh(plot_reco['개선추천'], plot_reco['recommendation_score'], color=colors, alpha=0.85)
ax.set_title(f'POINT_ID {selected_point_id} 개선우선순위 점수')
ax.set_xlabel('recommendation score')
plt.tight_layout()
plt.show()